In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

#### I'll be testing mostly **BERT** (encoder only) models for layer 1
Because
- Their tokens consist of subwords which means they can pay more attention to details within words themselves
- They are great for tasks like classification, sentence similarity and embeddings unlike gpt models which are good for sentence generation
- each word can see the previous and next words unlike gpt which goes from left to right only
- GPT ones are not suitable because they are more biased towards generating and what the next word will be

## Below is the difference between the tokens and embeddings generated from both GPT-2 and BERT models

In [15]:
# Create some sentences 
# i created a command sentence because well this is what we will usually use 
# and i created a normal sentence to show the difference between the models more clearly
command = "git merge feature/ai_pipeline"
sentence = "FancyGit is a tool used to help beginner developers to understand git and loosen the friction"

# -------- BERT MODEL -----------

# this is the basic bert model 
# (12 transformer encoder blocks, 768 hidden size, 12 attention heads)
bert_model_name = 'bert-base-uncased'  

# load both tokenizer and model
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

# prepare bert_model for inference only using .eval()
bert_model = AutoModel.from_pretrained(bert_model_name)
bert_model.eval()

# tokenize the sentence using bert_tokenizer
bert_tokens = bert_tokenizer(sentence, return_tensors = 'pt')    # return tensors in the form of pytorch tensors
bert_tokens_as_words = bert_tokenizer.convert_ids_to_tokens(bert_tokens['input_ids'][0])
# do inference and fetch the output
with torch.no_grad():    # <--- this will check all "requires_grad" parameters and set them to false (useful for inference)
    bert_outputs = bert_model(**bert_tokens)     # **tokens to unpack input_ids and attention_mask this will return embeddings

# fetch both sentence and token embeddings
bert_embeddings, bert_sentence_embedding = bert_outputs['last_hidden_state'], bert_outputs['pooler_output']


# -------- GPT MODEL -----------

# this is the GPT-2 model
# (12 transformer decoder blocks, 768 hidden size, 12 attention heads)
gpt_model_name = 'gpt2'

gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_model_name)
gpt_model = AutoModel.from_pretrained(gpt_model_name)
gpt_model.eval()

gpt_tokens = gpt_tokenizer(sentence, return_tensors = 'pt')
gpt_tokens_as_words = gpt_tokenizer.convert_ids_to_tokens(gpt_tokens['input_ids'][0])    # get tokens of first sentence

with torch.no_grad():
    gpt_outputs = gpt_model(**gpt_tokens)

print(gpt_outputs)  # AS YOU CAN SEE THERE IS NO CLS OR POOLER OUTPUTS AKA THERE IS NO SENTENCE EMBEDDINGS
# IT IS ALWAYS MORE FOCUSED ON GENERATING THE NEXT WORD
# which is one of the reason bert dominates in our project

gpt_embeddings = gpt_outputs['last_hidden_state']

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=tensor([[[-0.0748, -0.1390, -0.1788,  ..., -0.2495,  0.0763, -0.0440],
         [ 0.2487, -0.0153, -0.2860,  ..., -0.3282,  0.1952,  0.2010],
         [-1.0360, -0.1207, -0.1277,  ..., -0.2359,  0.4660, -0.1822],
         ...,
         [-1.2566,  0.7646, -1.4825,  ..., -0.9501, -0.3368, -0.6535],
         [-0.6550,  0.3605, -0.7428,  ..., -0.7533,  0.0542, -0.3445],
         [ 0.1933,  0.4565, -2.1059,  ..., -0.4292, -0.2549, -0.4734]]]), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, attentions=None, cross_attentions=None)


In [ ]:
print(bert_tokens_as_words)     # as u can see there is an extra cls token in bert model which helps in identifying sentence location

['[CLS]', 'fancy', '##git', 'is', 'a', 'tool', 'used', 'to', 'help', 'begin', '##ner', 'developers', 'to', 'understand', 'gi', '##t', 'and', 'loosen', 'the', 'friction', '[SEP]']


In [18]:
print(gpt_tokens_as_words)

['F', 'ancy', 'G', 'it', 'Ġis', 'Ġa', 'Ġtool', 'Ġused', 'Ġto', 'Ġhelp', 'Ġbeginner', 'Ġdevelopers', 'Ġto', 'Ġunderstand', 'Ġgit', 'Ġand', 'Ġloosen', 'Ġthe', 'Ġfriction']


In [ ]:
print(bert_embeddings.shape)        # which explains why the embeddings are more than the number of actual words in the sentence (16)
print(gpt_embeddings.shape)

torch.Size([1, 21, 768])
torch.Size([1, 19, 768])


In [ ]:
bert_similarity_matrix_friction = cosine_similarity(bert_embeddings[0, 19].unsqueeze(0), bert_embeddings[0])  # the word "friction" with all other words
gpt_similarity_matrix_friction = cosine_similarity(gpt_embeddings[0, 18].unsqueeze(0), gpt_embeddings[0])    # to convert it to a [1, 768]

In [ ]:
bert_similarity_matrix_friction

array([[0.22531301, 0.28812233, 0.43025374, 0.35883933, 0.38463217,
        0.41963226, 0.34177375, 0.34297422, 0.36392295, 0.34943688,
        0.41582382, 0.4409364 , 0.37094504, 0.3361606 , 0.36194414,
        0.48752022, 0.33473465, 0.48811224, 0.46624732, 0.9999999 ,
        0.09963042]], dtype=float32)

In [ ]:
gpt_similarity_matrix_friction

array([[0.952031  , 0.9938141 , 0.9786489 , 0.9861765 , 0.98515   ,
        0.97440875, 0.99173754, 0.98339385, 0.9480616 , 0.9663241 ,
        0.979398  , 0.96606207, 0.95702094, 0.96909964, 0.99427897,
        0.9697088 , 0.99125093, 0.9847208 , 1.0000004 ]], dtype=float32)

In [ ]:
bert_similarity_matrix = cosine_similarity(bert_embeddings[0])
gpt_similarity_matrix = cosine_similarity(gpt_embeddings[0])



##### as we can see BERT models are better and more accurate because they can look both ways bidirectional while GPT ones are more onto looking forward only which is the cosine_similarity is mostly 1 for all with last word